# 41b · **Bậc 1** — fine-tune CNN trên CXR-LT, đo lại `corr`, blend miễn phí

## Câu hỏi duy nhất của bậc này

`41c` đo `corr(CNN, RAD-DINO) = 0.5183` ở **mức probe** (đặc trưng đông cứng). Nhưng fine-tune trên
**nhãn đích** kéo **mọi** mô hình về cùng cấu trúc nhãn — lực kéo mạnh hơn hẳn pretrain PadChest.
Nên `0.5183` là ước lượng **lạc quan**, và con số thật chỉ có sau khi fine-tune.

```
cong  corr sau fine-tune < 0.75         ->  len bac 2 (dung CA kien truc tren backbone CNN)
      0.75 - 0.82                       ->  van len, nhung ky vong giam manh
      > 0.82  (~MOC same-backbone 0.8236) ->  DONG. Da dang bien mat.
```

Bậc này **cố tình rẻ**: đầu đọc đơn giản (global-pool → Linear), không `Tier1`, không `Tier1Img`.
Mục đích là **giết hướng sớm nếu nó phải chết**, không phải để có thành viên tốt.

## Và một con số thật, miễn phí

Sau khi có dự đoán của CNN, blend nó vào **đường ống thật** (`t1_prior_both` đã ensemble theo giao
thức `58`) với trọng số chọn trên **lát chọn**. Cho biết ngay hướng này đáng bao nhiêu trước khi
tiêu 6 giờ cho bậc 2.

```
moc = 0.2631 (58: ensemble epoch `all` x seed)   KHONG phai probe 0.2042
cong blend >= +0.005 macro VA tail khong tut
```

`41c` C5 cho `+0.0097` nhưng trên nền `0.2042` — nền thấp hơn nhiều, và phần lớn khoảng cách đó
đường ống hiện tại **đã lấp rồi**. Kỳ vọng ở đây nhỏ hơn.

## Kỷ luật

**Trọng số chọn trên lát chọn, báo cáo trên val** — chọn trên val rồi báo cáo trên val là bẫy §13.
**Một trọng số toàn cục**, không per-label: 11 nhãn tail có 19–129 ca dương, chọn riêng từng nhãn
sẽ overfit chắc chắn.
**Ensemble tất cả epoch** của CNN (`58`: cửa sổ `all` thắng trên lát chọn).
**Trung bình xác suất** là mặc định — `41c` C5 đo prob thắng rank `+0.0058` ở đúng cặp này.

## C0 · Config

In [1]:
import os, json, math, time, re, glob, datetime, shutil
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    import timm
except ImportError:
    import subprocess, sys as _s
    subprocess.run([_s.executable, '-m', 'pip', 'install', '-q', 'timm']); import timm
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('CXRLT_DATA', '')
    assert DATA_DIR, 'khong phai Colab va chua dat CXRLT_DATA -> gan thang duong dan `planB_out`'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'r1'
RES41B = os.path.join(OUT_DIR, 'stage4_nb41b', RUN_TAG)
RES41C = os.path.join(OUT_DIR, 'stage4_nb41c')
RES49  = os.path.join(OUT_DIR, 'stage4_nb49')
P_TOUCH = os.path.join(OUT_DIR, '_test_touches_nb41b.jsonl')
# DriveFS da nuot ghi 6 lan trong du an (trong so 41c, hien vat s42 cua 53, 374/378 tar kho anh,
# ca thu muc stage4_nb41b — HAI lan). Nen /content la noi ghi CHINH; Drive chi la ban sao.
RES_LOC = '/content/nb41b'; os.makedirs(RES_LOC, exist_ok=True)
SEARCH = [RES_LOC, '/content/salvage41b', RES41B]      # thu tu doc: local truoc, Drive sau cung
def ensure_res():
    try:
        os.makedirs(RES41B, exist_ok=True)
    except Exception:
        pass                                   # mount hong -> ke, /content moi la noi ghi chinh
    return RES41B
def put(name, fn):
    '''Ghi LOCAL truoc (chac chan), roi sao len Drive. KHONG bao gio ghi thang len mount.
       Mount hong thi chi canh bao — ban local van du de chay tiep.'''
    lp = os.path.join(RES_LOC, name); fn(lp)
    try:
        ensure_res(); shutil.copy(lp, os.path.join(RES41B, name))
        if not os.path.exists(os.path.join(RES41B, name)): raise OSError('file khong hien ra')
    except Exception as e:
        print(f'  !! sao {name} len Drive that bai ({type(e).__name__}); ban local o {lp}')
    return lp
def find(pat):
    '''Gom file theo TEN, uu tien thu muc dau tien tim thay.'''
    seen = {}
    for d in SEARCH:
        for f in sorted(glob.glob(os.path.join(d, pat))):
            seen.setdefault(os.path.basename(f), f)
    return [seen[k] for k in sorted(seen)]
ensure_res()
assert os.path.isdir(OUT_DIR) and os.listdir(OUT_DIR), f'{OUT_DIR} khong doc duoc -> mount hong'
LOCAL = '/content/cxrlt_518'
READY_X = os.path.join(OUT_DIR, 'cxrlt_518_ready')

# --- nguon duong ong de blend + do corr. Doi day neu dung lan chay khac ---
SRC_PIPE = [os.path.join(OUT_DIR, 'stage4_nb57', 'r3'),
            os.path.join(OUT_DIR, 'stage4_nb53', 'r3')]
ARM_PIPE = ('tpb', 't1_prior_both')      # ten arm trong 57 / trong 53 — cung kien truc
ARM_IMG  = 'mldec'                       # arm ANH-THUAN tren backbone `ft` -> corr backbone thuan

MODEL = 'convnextv2_nano.fcmae_ft_in22k_in1k'
P_BB  = os.path.join(RES41C, 'convnextv2_padchest.pt')
SEED, SEEDS = 42, (42,)                  # BAC 1 la CONG, khong phai so cuoi -> mot seed du
SIZE, BATCH, ACCUM = 518, 96, 3          # hieu dung 288; nano @518 ~13 GMAC nen batch lon duoc
# E=8. Rut ngan E KHONG cham duoc overfit — chi lam cosine ha LR som hon, dinh van o ep4 va
# THAP hon (do: E=8 dinh .2149, E=5 dinh .2128). Va giao thuc 58 ensemble TAT CA epoch, nen
# cac epoch sau dinh van dong gop DA DANG du yeu hon.
EPOCHS, LR, WD, WARMUP, CLIP = 8, 1e-4, 0.05, 0.05, 5.0   # fine-tune: LR nho, chuan LP-FT
NW = min(16, (os.cpu_count() or 2))
DO_TEST = True
MOC = 0.8236                             # 49: corr same-backbone — nguong "da dang bien mat"
GATE_CORR, GATE_BLEND = 0.75, 0.005

# Chan spam `Exception ignored in __del__: can only test a child process` cua DataLoader.
# No KHONG di qua `warnings` (nen `filterwarnings` vo dung) — Python bao cao qua `unraisablehook`.
# Loc DUNG loai nay, giu moi loi khac hien ra: che tat ca la tu bit mat.
import sys as _sys
_old_unraisable = _sys.unraisablehook
def _quiet_unraisable(u):
    if isinstance(u.exc_value, AssertionError) and 'child process' in str(u.exc_value): return
    _old_unraisable(u)
_sys.unraisablehook = _quiet_unraisable

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

_bb_loc = '/content/convnextv2_padchest.pt'
BB_USE = P_BB if os.path.exists(P_BB) else (_bb_loc if os.path.exists(_bb_loc) else None)
assert BB_USE, (f'khong thay trong so PadChest o CA {P_BB} lan {_bb_loc} -> chay 41c C3 truoc')
print(f'trong so CNN: {BB_USE}')

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
YM = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
        .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL          # CUNG cach cat nhu 35..59
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,} | '
      f'test {len(I_TE):,}')
print(f'{MODEL} | E={EPOCHS} LR={LR} | BATCH {BATCH}x{ACCUM}={BATCH*ACCUM} | seeds {SEEDS}')

## C1 · Ảnh + nạp dự đoán đường ống

Đường ống được ensemble theo đúng giao thức `58`: **tất cả** epoch × mọi seed có. Dừng ồn ào kèm
danh sách nếu không tìm thấy — bài học `41c`.

In [2]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from torchvision import transforms as T2
import tarfile, io as _io, shutil

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh dang co: {_n:,} / {len(ORDER):,}', flush=True)
if _n < len(ORDER) * 0.999 and os.path.isdir(READY_X):
    _kt = sorted(x for x in os.listdir(READY_X) if x.endswith('.tar'))
    print(f'kho co {len(_kt)} tar -> hydrate', flush=True)
    for _i, f in enumerate(_kt):
        src = os.path.join(READY_X, f); tmp = os.path.join('/content', f)
        try:
            shutil.copy(src, tmp)                     # copy TRUOC — khong mo tar tren mount FUSE
            assert os.path.getsize(tmp) == os.path.getsize(src), 'copy NGAN tu Drive'
            with tarfile.open(tmp) as tar: tar.extractall(LOCAL, filter='data')
        except Exception as e:
            print(f'  {f} HONG ({type(e).__name__}) -> bo qua')
        if os.path.exists(tmp): os.remove(tmp)
        print(f'  hydrate {_i+1}/{len(_kt)} {f}', flush=True)
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
# --- chua du thi stage tu shard goc, VA ghi vao kho de lan sau khoi lap ---
# Kho `cxrlt_518_ready` co the thieu (Drive danh roi tar, hoac lan tao truoc chua chay het).
# Doi hoi nguoi dung di copy cell tu notebook khac la mot phu thuoc NGAM — bai hoc 41c.
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
    assert os.path.isdir(SHARD_DIR), f'khong thay CA kho {READY_X} lan shard goc {SHARD_DIR}'
    print('doc danh sach shard tren Drive...', flush=True)
    TARS = sorted(f for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    def _readyx():
        os.makedirs(READY_X, exist_ok=True); return READY_X       # FUSE danh roi thu muc RONG
    _tarof = lambda t: os.path.join(READY_X, 'x518_' + os.path.splitext(t)[0] + '.tar')
    done = {t for t in TARS if os.path.exists(_tarof(t))}          # TAR co that = nguon su that
    print(f'{len(TARS)} shard | da co trong kho {len(done)}')
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return dst
        try:
            im = Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC)
        except Exception:
            return None                                  # mot anh hong KHONG giet ca lan chay
        tmp = dst + '.tmp'; im.save(tmp, 'JPEG', quality=95); os.replace(tmp, dst)   # NGUYEN TU
        return dst
    _t0 = time.time()
    with ThreadPoolExecutor(16) as pool:
        for t in TARS:
            if t in done: continue
            tp = os.path.join(SHARD_DIR, t)
            with open(tp, 'rb') as f: buf = f.read()     # doc BULK; lap tren mount FUSE se TREO
            assert len(buf) == os.path.getsize(tp), f'{t}: doc NGAN tren mount, chay lai cell'
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            del buf
            made = [p for p in pool.map(_one, jobs) if p]; del jobs
            if made:                                     # ghi kho NGAY: dut giua chung mat 1 shard
                _tp = os.path.join('/content', os.path.basename(_tarof(t)))
                with tarfile.open(_tp, 'w') as tar:
                    for p in made: tar.add(p, arcname=os.path.basename(p))
                shutil.copy(_tp, _tarof(t)); os.remove(_tp)
                # DOC LAI kiem: Drive da nuot ghi 3 lan trong du an (trong so 41c, hien vat s42
                # cua 53, va chinh kho nay - 374/378 tar bien mat). Bao NGAY tai shard hong.
                if not os.path.exists(_tarof(t)) or os.path.getsize(_tarof(t)) == 0:
                    print(f'  !! {os.path.basename(_tarof(t))} KHONG hien ra tren Drive '
                          '-> kho se thieu, nhung anh local van du. Chay tiep.', flush=True)
            _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
            print(f'  {t:<18} tong {_n:>7,} | {(time.time()-_t0)/60:5.1f} ph', flush=True)
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'moi co {_n:,}/{len(ORDER):,} anh sau khi stage'
print(f'anh local: {_n:,} / {len(ORDER):,}')

def load_pipe():
    '''Ensemble TAT CA epoch x seed cho mot arm — giao thuc 58 (cua so `all` thang tren lat chon).'''
    out = {}
    for arm in list(ARM_PIPE) + [ARM_IMG]:
        for d in SRC_PIPE:
            fv = sorted(glob.glob(os.path.join(d, f'p_{arm}_s*_ep*.npy')))
            fs = sorted(glob.glob(os.path.join(d, f'psl_{arm}_s*_ep*.npy')))
            if not (fv and fs): continue
            pv = np.mean([np.load(f).astype('float32') for f in fv], 0)
            ps = np.mean([np.load(f).astype('float32') for f in fs], 0)
            assert pv.shape == (len(I_VA), C) and ps.shape == (len(I_SL), C), \
                f'{arm}@{d}: hinh dang {pv.shape}/{ps.shape}'
            out[arm] = dict(val=pv, sel=ps, n=len(fv), src=d)
            break
    return out
PIPE = load_pipe()
if not PIPE:
    print('KHONG nap duoc arm nao. Cac file p_*.npy dang co:')
    for d in SRC_PIPE:
        for f in sorted(glob.glob(os.path.join(d, 'p_*_s*_ep*.npy')))[:6]:
            print('   ' + f.replace(OUT_DIR, '...'))
assert PIPE, ('khong nap duoc du doan duong ong nao -> doi SRC_PIPE/ARM_PIPE o C0 cho khop '
              'RUN_TAG that su co tren dia.')
PIPE_ARM = next(a for a in ARM_PIPE if a in PIPE)
for a, v in PIPE.items():
    print(f'  {a:<16} {v["n"]:>3} file  <- {v["src"].replace(OUT_DIR, "...")}')
print(f'  -> arm duong ong dung de blend: `{PIPE_ARM}`'
      + (f' | arm anh-thuan de do corr backbone: `{ARM_IMG}`' if ARM_IMG in PIPE else
         ' | KHONG co arm anh-thuan -> bo qua corr backbone'))

anh dang co: 0 / 377,110
kho co 382 tar -> hydrate
  hydrate 1/382 x518_lb_00000 (1).tar
  hydrate 2/382 x518_lb_00000.tar
  hydrate 3/382 x518_lb_00001 (1).tar
  hydrate 4/382 x518_lb_00001.tar
  hydrate 5/382 x518_lb_00002 (1).tar
  hydrate 6/382 x518_lb_00002.tar
  hydrate 7/382 x518_lb_00003 (1).tar
  hydrate 8/382 x518_lb_00003.tar
  hydrate 9/382 x518_lb_00004.tar
  hydrate 10/382 x518_lb_00005.tar
  hydrate 11/382 x518_lb_00006.tar
  hydrate 12/382 x518_lb_00007.tar
  hydrate 13/382 x518_lb_00008.tar
  hydrate 14/382 x518_lb_00009.tar
  hydrate 15/382 x518_lb_00010.tar
  hydrate 16/382 x518_lb_00011.tar
  hydrate 17/382 x518_lb_00012.tar
  hydrate 18/382 x518_lb_00013.tar
  hydrate 19/382 x518_lb_00014.tar
  hydrate 20/382 x518_lb_00015.tar
  hydrate 21/382 x518_lb_00016.tar
  hydrate 22/382 x518_lb_00017.tar
  hydrate 23/382 x518_lb_00018.tar
  hydrate 24/382 x518_lb_00019.tar
  hydrate 25/382 x518_lb_00020.tar
  hydrate 26/382 x518_lb_00021.tar
  hydrate 27/382 x518_lb_00022.t

## C2 · Chấm điểm + dataset + fine-tune

In [3]:
from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
mac = lambda Y_, P_: float(np.nanmean(ap_cols(Y_, P_)))
tie = lambda Y_, P_: {t: float(np.nanmean(ap_cols(Y_, P_)[i])) for t, i in TIER.items()}
_r6 = np.random.default_rng(6); _y6 = (_r6.random((3000, C)) < 0.2).astype('float32')
_s6 = _r6.random((3000, C)).astype('float32')
assert np.nanmax(np.abs(ap_cols(_y6, _s6) - np.array(
    [average_precision_score(_y6[:, j], _s6[:, j]) for j in range(C)]))) < 1e-6
print('cong 1: ap_cols khop sklearn')

class ASL(nn.Module):
    '''Verbatim nb11..59. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()

_m0 = timm.create_model(MODEL, pretrained=True, num_classes=0, in_chans=1)
_cfg = timm.data.resolve_model_data_config(_m0)
MU, SD = float(np.mean(_cfg['mean'])), float(np.mean(_cfg['std']))
_IN_SD = {k: v.clone() for k, v in _m0.state_dict().items()}; del _m0
tf_tr = T2.Compose([T2.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                         interpolation=T2.InterpolationMode.BICUBIC),
                    T2.RandomRotation(7)])

class DS(Dataset):
    '''Anh xam 1 kenh, chuan hoa bang mean/std cua CHINH model — khop 41c C4.'''
    def __init__(s, idx, train): s.idx, s.train = np.asarray(idx), train
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k])
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        if s.train: im = tf_tr(im)
        x = torch.from_numpy(np.asarray(im, dtype='uint8').copy()).unsqueeze(0).float().div_(255)
        return (x - MU) / SD, torch.from_numpy(YM[i]), k

def loader(idx, train, bs, nw=NW):
    return DataLoader(DS(idx, train), batch_size=bs, shuffle=train, num_workers=nw,
                      pin_memory=True, drop_last=train, persistent_workers=nw > 0,
                      prefetch_factor=4 if nw > 0 else None)

# Loader EVAL tao MOT LAN roi dung lai. Tao moi moi epoch voi `num_workers>0` +
# `persistent_workers` lam PyTorch goi `is_alive()` tren worker khong phai con cua tien trinh
# dang thu gom -> ngap man hinh `Exception ignored in __del__: can only test a child process`.
# Vo hai (exception BI BO QUA) nhung che het log, va tao/huy 16 worker moi epoch cung ton thoi gian.
_DLC = {}
def eval_loader(idx, bs):
    k = (len(idx), int(idx[0]), int(idx[-1]), bs)
    if k not in _DLC: _DLC[k] = loader(idx, False, bs)
    return _DLC[k]

def build_net():
    net = timm.create_model(MODEL, pretrained=False, num_classes=C, in_chans=1)
    ck = torch.load(BB_USE, map_location='cpu', weights_only=False)
    miss = net.load_state_dict({k: v for k, v in ck['sd'].items() if not k.startswith('head.fc.')},
                               strict=False)
    bad = [k for k in miss.missing_keys if not k.startswith('head.fc.')]
    assert not bad, f'thieu khoa khi nap backbone PadChest: {bad[:5]}'
    return net.to(dev), ck

_net0, _ck0 = build_net()
# cong 2: trong so PHAI khac ImageNet — nap nham ban goc thi moi so ve `corr` va do manh vo nghia
with torch.no_grad():
    _d = max((_IN_SD[k] - v.detach().cpu()).abs().max().item()
             for k, v in _net0.state_dict().items() if k in _IN_SD)
assert _d > 1e-4, (f'trong so GIONG HET ImageNet (lech {_d:.1e}) -> `convnextv2_padchest.pt` khong '
                   'duoc nap. Moi so ve do manh/corr se vo nghia.')
print(f'cong 2: trong so PadChest da nap | lech ImageNet {_d:.3f} | '
      f'ep {_ck0.get("ep", "?")} mAP PadChest {_ck0.get("mAP", float("nan")):.4f}')
del _net0

cong 1: ap_cols khop sklearn


model.safetensors: reconstructing file:   0%|          |  0.00B / 62.5MB            

model.safetensors: downloading bytes:           |  0.00B            

cong 2: trong so PadChest da nap | lech ImageNet 1.558 | ep 8 mAP PadChest 0.3600


## C3 · Fine-tune trên CXR-LT — **epoch cuối**, lưu dự đoán từng epoch

Cosine trải đúng `E`, không early-stop (`35` §0.1). Lưu `p_*`/`psl_*` mỗi epoch để C4 ensemble theo
giao thức `58` và để `E` đọc được từ đường `LAT CHON`.

In [4]:
import shutil
crit = ASL(gn=4, gp=0).to(dev)
sig = lambda x: 1.0 / (1.0 + np.exp(-x))

@torch.no_grad()
def predict(net, idx, bs=256):
    net.eval(); P = np.zeros((len(idx), C), 'float32')
    for x, _, k in tqdm(eval_loader(idx, bs), desc='eval', leave=False):
        with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
            lg = net(x.to(dev, non_blocking=True))
        P[k.numpy()] = torch.sigmoid(lg.float()).cpu().numpy()
    assert np.isfinite(P).all() and np.abs(P).sum(1).min() > 0, 'predict cho ra NaN/hang trong'
    return P

P_LOG = os.path.join(RES_LOC, 'epoch_log.json')     # nhat ky cung ghi LOCAL truoc
_old_log = find('epoch_log.json')
LOG = json.load(open(_old_log[0])) if _old_log else {}
for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'): print(f'{sk}: da xong, bo qua'); continue
    torch.manual_seed(seed)
    net, _ = build_net()
    opt = torch.optim.AdamW([{'params': [p for p in net.parameters() if p.ndim > 1],
                              'weight_decay': WD},
                             {'params': [p for p in net.parameters() if p.ndim <= 1],
                              'weight_decay': 0.0}], lr=LR)
    steps = math.ceil(len(I_TF) / (BATCH*ACCUM)) * EPOCHS; warm = int(WARMUP*steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    dl = loader(I_TF, True, BATCH)
    LOG.setdefault(sk, {}); nclip = 0; nstep = 0
    for ep in range(EPOCHS):
        net.train(); t0, tot, nb = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for j, (x, y, _) in enumerate(tqdm(dl, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
                lg = net(x.to(dev, non_blocking=True))
            l_ = crit(lg.float(), y.to(dev))
            (l_ / ACCUM).backward(); tot += float(l_.detach()); nb += 1
            if (j + 1) % ACCUM == 0:
                gn = torch.nn.utils.clip_grad_norm_(net.parameters(), CLIP,
                                                    error_if_nonfinite=False)
                if not torch.isfinite(gn):
                    opt.zero_grad(set_to_none=True); continue   # buoc hong -> BO, dung boi ban
                nclip += int(float(gn) > CLIP); nstep += 1
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        PV, PS = predict(net, I_VA), predict(net, I_SL)
        # Ghi LOCAL truoc roi copy — va goi `ensure_res()` NGAY TRUOC tung lan copy.
        # Lan truoc chet vi `np.save(path, predict(...))`: duong dan tinh ngay nhung `predict`
        # chay vai phut KHONG ghi gi len Drive, va FUSE danh roi thu muc trong dung cua so do.
        put(f'p_cnn_{sk}_ep{ep+1}.npy', lambda p: np.save(p, PV.astype('float16')))
        put(f'psl_cnn_{sk}_ep{ep+1}.npy', lambda p: np.save(p, PS.astype('float16')))
        if DO_TEST and ep == EPOCHS - 1:
            _pte = predict(net, I_TE).astype('float16')     # tinh XONG roi moi dung toi file
            put(f'pte_cnn_{sk}_ep{ep+1}.npy', lambda p: np.save(p, _pte)); del _pte
            with open(P_TOUCH, 'a') as _f:
                _f.write(json.dumps(dict(nb='41b', run=RUN_TAG, seed=seed, ep=ep+1, n=len(I_TE),
                                         muc_dich='theo doi',
                                         t=datetime.datetime.now().isoformat(timespec='seconds')))+'\n')
        ap = ap_cols(YM[VA], PV); t = tie(YM[VA], PV)
        LOG[sk][str(ep)] = dict(mAP_anh=float(np.nanmean(ap)), **t,
                                sel=mac(YM[SL], PS), clip_pct=nclip/max(nstep, 1))
        put('epoch_log.json', lambda p: json.dump(LOG, open(p, 'w')))
        print(f'{sk} ep{ep+1:>2}/{EPOCHS} | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | '
              f'{(time.time()-t0)/60:.1f} ph | loss {tot/max(nb,1):.4f} | '
              f'val {np.nanmean(ap):.4f} | LAT CHON {mac(YM[SL], PS):.4f} | tail {t["tail"]:.4f}')
    put(f'cnn_ft_{sk}.pt',
        lambda p: torch.save(dict(sd=net.state_dict(), seed=seed, model=MODEL), p))
    LOG[sk]['done'] = True; put('epoch_log.json', lambda p: json.dump(LOG, open(p, 'w')))
    del net; torch.cuda.empty_cache()
print('fine-tune xong')

s42 ep1/5:   0%|          | 0/2289 [00:00<?, ?it/s]

eval:   0%|          | 0/154 [00:00<?, ?it/s]

eval:   0%|          | 0/153 [00:00<?, ?it/s]

s42 ep 1/5 | 202 img/s | 18.2 ph | loss 0.0438 | val 0.1979 | LAT CHON 0.1868 | tail 0.0243


s42 ep2/5:   0%|          | 0/2289 [00:00<?, ?it/s]

...

s42 ep5/5:   0%|          | 0/2289 [00:00<?, ?it/s]

eval:   0%|          | 0/154 [00:00<?, ?it/s]

eval:   0%|          | 0/153 [00:00<?, ?it/s]

eval:   0%|          | 0/309 [00:00<?, ?it/s]

s42 ep 5/5 | 187 img/s | 19.6 ph | loss 0.0377 | val 0.2195 | LAT CHON 0.2117 | tail 0.0468
fine-tune xong


## C4 · **CỔNG `corr`** + blend với trọng số chọn trên lát chọn

In [5]:
from scipy.stats import rankdata
def corr(A, B):
    ra, rb = rankdata(A, axis=0), rankdata(B, axis=0)
    ra = (ra - ra.mean(0)) / (ra.std(0) + 1e-12); rb = (rb - rb.mean(0)) / (rb.std(0) + 1e-12)
    return float(np.mean((ra * rb).mean(0)))

_fv = find('p_cnn_s*_ep*.npy')
_fs = find('psl_cnn_s*_ep*.npy')
if not (_fv and _fs):
    for d in SEARCH:
        print(f'  {d}: ' + (', '.join(sorted(os.listdir(d))[:8]) if os.path.isdir(d)
                            else 'KHONG TON TAI'))
assert _fv and _fs, f'khong thay du doan CNN trong {SEARCH} -> chay C3 truoc'
print(f'nguon: ' + ', '.join(sorted({os.path.dirname(f) for f in _fv})))
CNN_V = np.mean([np.load(f).astype('float32') for f in _fv], 0)   # ensemble TAT CA epoch (58)
CNN_S = np.mean([np.load(f).astype('float32') for f in _fs], 0)
print(f'CNN: ensemble {len(_fv)} file (tat ca epoch x seed)')

PV, PS = PIPE[PIPE_ARM]['val'], PIPE[PIPE_ARM]['sel']
M_CNN, M_PIPE = mac(YM[VA], CNN_V), mac(YM[VA], PV)
CORR_PIPE = corr(CNN_V, PV)
CORR_IMG = corr(CNN_V, PIPE[ARM_IMG]['val']) if ARM_IMG in PIPE else float('nan')
print(f'\n{"":<22}{"mAP":>9}{"corr voi CNN":>14}')
print(f'  {"CNN fine-tuned":<20}{M_CNN:>9.4f}{"—":>14}')
print(f'  {PIPE_ARM + " (duong ong)":<20}{M_PIPE:>9.4f}{CORR_PIPE:>14.4f}')
if ARM_IMG in PIPE:
    print(f'  {ARM_IMG + " (anh thuan)":<20}{mac(YM[VA], PIPE[ARM_IMG]["val"]):>9.4f}'
          f'{CORR_IMG:>14.4f}')
print(f'\n  MOC same-backbone (49) = {MOC:.4f} | 41c do corr o muc PROBE = 0.5183')

print('\n=== CONG 1: corr SAU fine-tune ===')
_c = CORR_IMG if ARM_IMG in PIPE else CORR_PIPE
_v1 = ('DAT -> len bac 2' if _c < GATE_CORR else
       'vung xam -> van len, ky vong giam manh' if _c < 0.82 else
       'DONG: da dang bien mat sau fine-tune')
print(f'  corr = {_c:.4f} (nguong {GATE_CORR}) -> {_v1}')

# --- blend: TRONG SO chon tren LAT CHON, bao cao tren VAL ---
WG = np.round(np.arange(0, 1.001, 0.05), 2)
_sel_sc = [mac(YM[SL], (1-w)*PS + w*CNN_S) for w in WG]
W = float(WG[int(np.argmax(_sel_sc))])
BL_V = (1-W)*PV + W*CNN_V
G_BL = mac(YM[VA], BL_V) - M_PIPE
T_BL = tie(YM[VA], BL_V)['tail'] - tie(YM[VA], PV)['tail']
print(f'\n=== CONG 2: blend (w chon tren LAT CHON) ===')
print(f'  w = {W:.2f}  (sel {max(_sel_sc):.4f} vs w=0 {_sel_sc[0]:.4f})')
print(f'  {"duong ong don le":<22}{M_PIPE:>9.4f}{tie(YM[VA], PV)["tail"]:>9.4f}')
print(f'  {"+ CNN (blend)":<22}{mac(YM[VA], BL_V):>9.4f}{tie(YM[VA], BL_V)["tail"]:>9.4f}')
print(f'  delta macro {G_BL:+.4f} | tail {T_BL:+.4f}   (cong >= +{GATE_BLEND})')
_w_eq = mac(YM[VA], 0.5*PV + 0.5*CNN_V) - M_PIPE
print(f'  (trung binh DEU se cho {_w_eq:+.4f} — vi sao khong dung trong so deu)')

print('\n' + '=' * 72)
_v2 = ('NHAN: dung bac 2' if (G_BL >= GATE_BLEND and T_BL >= -0.003 and _c < 0.82) else
       'DONG huong ensemble CNN' if _c >= 0.82 else
       'vung xam: blend duoi cong nhung corr con thap -> bac 2 co the cuu')
print(f'KET LUAN BAC 1: {_v2}')
print('=' * 72)
print('Nho: bac 1 dung dau doc DON GIAN (khong Tier1, khong Tier1Img). Bac 2 dung CA kien truc')
print('tren backbone CNN, nen thanh vien se manh hon HAN va blend co the vuot cong du bac 1 truot.')

L = [f'# REPORT · `nb41b` bac 1 — fine-tune `{MODEL}` tren CXR-LT', '',
     f'`{EPOCHS}` epoch · seeds `{list(SEEDS)}` · dau doc don gian · ensemble tat ca epoch (`58`).', '',
     '| | mAP | corr voi CNN |', '|---|---|---|',
     f'| CNN fine-tuned | {M_CNN:.4f} | — |',
     f'| `{PIPE_ARM}` duong ong | {M_PIPE:.4f} | {CORR_PIPE:.4f} |']
if ARM_IMG in PIPE:
    L.append(f'| `{ARM_IMG}` anh thuan | {mac(YM[VA], PIPE[ARM_IMG]["val"]):.4f} | {CORR_IMG:.4f} |')
L += ['', '```', f'corr o muc PROBE (41c)     0.5183', f'corr SAU fine-tune         {_c:.4f}',
      f'MOC same-backbone (49)     {MOC:.4f}', f'blend w={W:.2f}            {G_BL:+.4f} macro | '
      f'{T_BL:+.4f} tail', '```', '', f'**Cong 1** (`corr < {GATE_CORR}`): {_v1}',
      f'**Cong 2** (`blend >= +{GATE_BLEND}`): {G_BL:+.4f}', '', f'**Ket luan: {_v2}**', '',
      '## Nhung gi ket qua nay KHONG tra loi', '',
      '- **Dau doc don gian**, khong `Tier1`/`Tier1Img`. Bac 2 se manh hon han.',
      '- **Mot seed** — bac 1 la cong, khong phai so cuoi.',
      '- So test chi de theo doi; model chi thay 74% du lieu co nhan.']
_rp = put('REPORT_STAGE4_NB41B.md',
          lambda p: open(p, 'w', encoding='utf-8').write(chr(10).join(L)))
print(chr(10) + 'da ghi ' + _rp)


nguon: /content/nb41b
CNN: ensemble 5 file (tat ca epoch x seed)

                            mAP  corr voi CNN
  CNN fine-tuned         0.2202             —
  tpb (duong ong)        0.2627        0.7023
  mldec (anh thuan)      0.2526        0.7137

  MOC same-backbone (49) = 0.8236 | 41c do corr o muc PROBE = 0.5183

=== CONG 1: corr SAU fine-tune ===
  corr = 0.7137 (nguong 0.75) -> DAT -> len bac 2

=== CONG 2: blend (w chon tren LAT CHON) ===
  w = 0.25  (sel 0.2535 vs w=0 0.2511)
  duong ong don le         0.2627   0.1127
  + CNN (blend)            0.2654   0.1138
  delta macro +0.0027 | tail +0.0011   (cong >= +0.005)
  (trung binh DEU se cho -0.0015 — vi sao khong dung trong so deu)

KET LUAN BAC 1: vung xam: blend duoi cong nhung corr con thap -> bac 2 co the cuu
Nho: bac 1 dung dau doc DON GIAN (khong Tier1, khong Tier1Img). Bac 2 dung CA kien truc
tren backbone CNN, nen thanh vien se manh hon HAN va blend co the vuot cong du bac 1 truot.

da ghi /content/nb41b/REPORT_STAGE4_